# Kubeflow Pipelines — Iris 머신러닝 파이프라인 (v1 → v2 → v3)

기존 실습은 그대로 유지하면서, **v3에서 교차검증·평가 메트릭·혼동행렬·품질 검사**를 추가한 정리본입니다.

- **v1:** 학습과 평가를 하나의 Component에서 수행
- **v2:** 학습 / 평가 Component 분리, `Model` · `Dataset` Artifact 전달
- **v3:** 학습 데이터 5-Fold 교차검증 → 최종 테스트 Accuracy/F1 → 품질 기준 확인

**기존 성공 기록:** v1 Run `0e79df49-12ff-40a0-85a9-479cdf79591f` (`SUCCEEDED`), v2 Run `39b54d90-fdb5-43b9-81a6-b885dd19750b` (`SUCCEEDED`, Output `1.0`).

> 주의: `create_run_from_pipeline_package()` 셀을 다시 실행할 때마다 새로운 KFP Run이 제출됩니다. 이미 성공한 v1·v2의 실행 요청 셀은 필요할 때만 실행하세요. 인증 토큰은 파일로 마운트하여 사용하며, 내용은 출력하거나 GitHub에 업로드하지 않습니다.


## 0. KFP SDK 연결
이 Notebook은 Kubeflow의 KFP 접근 설정이 적용된 `kfp-notebook`에서 실행합니다.

In [1]:
import kfp 
from kfp import dsl, compiler 
from kfp.client.set_volume_credentials import (
    ServiceAccountTokenVolumeCredentials
) 

client = kfp.Client(
    host="http://ml-pipeline.kubeflow.svc.cluster.local:8888", 
    namespace="kubeflow-user-example-com", 
    credentials=ServiceAccountTokenVolumeCredentials() 
) 

print("KFP 연결 성공!") 


KFP 연결 성공!


/opt/conda/lib/python3.12/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


## 1. v1 — 단일 Component에서 학습과 평가
아래 코드는 이전 실습의 재현용입니다.

In [2]:
@dsl.component(
    base_image="python:3.11-slim", 
    packages_to_install=["scikit-learn==1.5.2"]
) 

def train_model() -> float: 
    from sklearn.datasets import load_iris 
    from sklearn.model_selection import train_test_split 
    from sklearn.ensemble import RandomForestClassifier 
    from sklearn.metrics import accuracy_score 

    # 1. Iris 데이터 불러오기 
    iris = load_iris() 

    # 2. 학습용과 평가용 데이터 분리 
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, 
        iris.target, 
        test_size=0.2, 
        random_state=42 
    ) 

    # 3. RandomForest 모델 생성 
    model = RandomForestClassifier( 
        n_estimators=100, 
        random_state=42
    ) 

    # 4. 모델 학습 
    model.fit(X_train, y_train) 

    # 5. 예측 및 정확도 평가 
    predictions = model.predict(X_test) 
    accuracy = accuracy_score(y_test, predictions) 

    print("모델 정확도:", accuracy) 

    return float(accuracy) 

In [3]:
@dsl.pipeline(name="iris-training-pipeline") 
def iris_pipeline() -> float: 
    train_task = train_model() 
    return train_task.output

In [4]:
compiler.Compiler().compile( 
    pipeline_func=iris_pipeline, 
    package_path="iris-training-pipeline.yaml"
) 

print("Iris Pipeline YAML 생성 성공!") 

Iris Pipeline YAML 생성 성공!


In [5]:
# 기존 실행 성공 기록을 유지하기 위해 기본값은 False입니다.
# 같은 파이프라인을 다시 실행하려면 True로 변경하세요.
RERUN_V1 = False

if RERUN_V1:
    run = client.create_run_from_pipeline_package( 
        pipeline_file="iris-training-pipeline.yaml", 
        experiment_name="iris-mlops-experiment"
    ) 

    print("RUN ID:", run.run_id) 
else:
    print("v1 기존 실행은 유지합니다. 중복 Run을 생성하지 않았습니다.")


v1 기존 실행은 유지합니다. 중복 Run을 생성하지 않았습니다.


In [6]:
run_id = "0e79df49-12ff-40a0-85a9-479cdf79591f"

run_detail = client.get_run(run_id=run_id)

print("Pipeline 상태:", run_detail.state)

Pipeline 상태: SUCCEEDED


## 2. v2 — 모델 Artifact를 별도 평가 Component에 전달
`train_model_v2`가 모델과 테스트 데이터를 MinIO 아티팩트 저장소에 출력하고, `evaluate_model`이 이를 입력받습니다.

In [7]:
@dsl.component(
    base_image="python:3.11-slim", 
    packages_to_install=["scikit-learn==1.5.2"]
) 

def train_model_v2(
    model_artifact: dsl.Output[dsl.Model], 
    test_data: dsl.Output[dsl.Dataset]
): 
    import joblib 
    import numpy as np 

    from sklearn.datasets import load_iris 
    from sklearn.model_selection import train_test_split 
    from sklearn.ensemble import RandomForestClassifier 

    # 1. Iris 데이터 불러오기 
    iris = load_iris() 

    X_train, X_test, y_train, y_test = train_test_split( 
        iris.data, 
        iris.target, 
        test_size=0.2, 
        random_state=42
    ) 

    # 3. RandomForest 모델 생성 
    model = RandomForestClassifier( 
        n_estimators=100, 
        random_state=42
    ) 

    # 4. 모델 학습 
    model.fit(X_train, y_train) 

    # 5. 학습한 모델 파일 저장 
    joblib.dump(model, model_artifact.path) 

    # 6. 테스트 데이터 저장 
    with open(test_data.path, "wb") as f: 
        np.savez(
            f, 
            X_test=X_test, 
            y_test=y_test
        ) 

    print("RandomForest 학습 완료!") 
    print("학습 데이터:", len(X_train)) 
    print("테스트 데이터:", len(X_test)) 
    print("모델 파일 저장 완료!") 

In [8]:
@dsl.component(
    base_image="python:3.11-slim", 
    packages_to_install=["scikit-learn==1.5.2"]
) 
def evaluate_model(
    model_artifact: dsl.Input[dsl.Model], 
    test_data: dsl.Input[dsl.Dataset]
) -> float: 
    import joblib 
    import numpy as np 
    from sklearn.metrics import accuracy_score 

    # 1. 앞 단계에서 지정한 모델 불러오기 
    model = joblib.load(model_artifact.path) 

    # 2. 테스트 데이터 불러오기 
    with np.load(test_data.path) as data: 
        X_test = data["X_test"]
        y_test = data["y_test"]

    # 3. 모델 예측 
    predictions = model.predict(X_test) 

    # 4. 정확도 계산 
    accuracy = accuracy_score(y_test, predictions) 

    print("모델 정확도:", accuracy) 

    return float(accuracy) 


In [9]:
@dsl.pipeline(name="iris-train-evaluate-pipeline") 
def iris_train_evaluate_pipeline() -> float: 

    # 1. 모델 학습 작업 
    train_task = train_model_v2() 

    # 2. 모델 평가 작업 
    evaluate_task = evaluate_model( 
        model_artifact=train_task.outputs["model_artifact"], 
        test_data=train_task.outputs["test_data"]
    ) 

    # 3. 최종 적확도 반환 
    return evaluate_task.output 

In [10]:
compiler.Compiler().compile( 
    pipeline_func=iris_train_evaluate_pipeline, 
    package_path="iris-train-evaluate-pipeline.yaml"
) 

print("2단계 파이프라인 YAML 생성 성공!") 

2단계 파이프라인 YAML 생성 성공!


In [11]:
# 기존 실행 성공 기록을 유지하기 위해 기본값은 False입니다.
# 같은 파이프라인을 다시 실행하려면 True로 변경하세요.
RERUN_V2 = False

if RERUN_V2:
    run_v2 = client.create_run_from_pipeline_package( 
        pipeline_file="iris-train-evaluate-pipeline.yaml", 
        experiment_name="iris-mlops-experiment"
    ) 

    print("Run ID:", run_v2.run_id)
else:
    print("v2 기존 실행은 유지합니다. 중복 Run을 생성하지 않았습니다.")


v2 기존 실행은 유지합니다. 중복 Run을 생성하지 않았습니다.


## 3. v3 — 모델 평가 개선

v3는 v2를 변경하지 않고 별도의 세 작업으로 작성합니다.

1. `train_model_v3`: **훈련 데이터에만** 5-Fold 교차검증, 최종 모델 학습 후 Model·Dataset·Metrics 출력
2. `evaluate_model_v3`: 홀드아웃 테스트에서 Accuracy, macro F1, Confusion Matrix 계산
3. `quality_gate`: 교차검증 평균과 테스트 Accuracy가 지정 기준 이상인지 확인

**중요:** 30개 테스트 샘플에서 Accuracy 1.0이 나온 것이 전체 데이터에서도 100% 성능이라는 뜻은 아닙니다. CV 평균·편차와 테스트 성능을 함께 기록해 결과를 더 신중하게 해석합니다.


In [12]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["scikit-learn==1.5.2"]
)
def train_model_v3(
    model_artifact: dsl.Output[dsl.Model],
    test_data: dsl.Output[dsl.Dataset],
    train_metrics: dsl.Output[dsl.Metrics],
    n_estimators: int = 100
) -> float:
    import joblib
    import numpy as np

    from sklearn.datasets import load_iris
    from sklearn.model_selection import (
        train_test_split,
        StratifiedKFold,
        cross_val_score
    )
    from sklearn.ensemble import RandomForestClassifier

    # 1. 데이터 로드
    iris = load_iris()

    # 2. 학습 / 테스트 데이터 분리
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=42,
        stratify=iris.target
    )

    # 3. 모델 생성
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=42
    )

    # 4. 5-Fold 교차검증 (학습 데이터만 사용)
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print("교차검증 Accuracy:", scores)
    print("교차검증 평균:", scores.mean())
    print("교차검증 표준편차:", scores.std())

    # 5. 전체 학습 데이터로 최종 모델 학습
    model.fit(X_train, y_train)

    # 6. 모델 저장
    joblib.dump(model, model_artifact.path)

    model_artifact.metadata["framework"] = "scikit-learn"
    model_artifact.metadata["n_estimators"] = n_estimators

    # 7. 테스트 데이터 저장
    with open(test_data.path, "wb") as f:
        np.savez(
            f,
            X_test=X_test,
            y_test=y_test
        )

    # 8. KFP Metrics 기록
    train_metrics.log_metric(
        "cv_accuracy_mean",
        float(scores.mean())
    )

    train_metrics.log_metric(
        "cv_accuracy_std",
        float(scores.std())
    )

    # quality_gate에 교차검증 평균을 전달합니다.
    return float(scores.mean())

### 3-1. 최종 테스트 평가 Component
학습 단계가 저장한 모델과 테스트 데이터를 입력받아 독립적으로 평가합니다.

In [13]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["scikit-learn==1.5.2"]
)
def evaluate_model_v3(
    model_artifact: dsl.Input[dsl.Model],
    test_data: dsl.Input[dsl.Dataset],
    evaluation_metrics: dsl.Output[dsl.Metrics],
    classification_metrics: dsl.Output[dsl.ClassificationMetrics]
) -> float:
    import joblib
    import numpy as np
    from sklearn.metrics import (
        accuracy_score, f1_score, precision_score,
        recall_score, confusion_matrix
    )

    model = joblib.load(model_artifact.path)
    with np.load(test_data.path, allow_pickle=False) as data:
        X_test = data["X_test"]
        y_test = data["y_test"]

    predictions = model.predict(X_test)
    accuracy = float(accuracy_score(y_test, predictions))
    f1_macro = float(f1_score(y_test, predictions, average="macro", zero_division=0))
    precision_macro = float(precision_score(y_test, predictions, average="macro", zero_division=0))
    recall_macro = float(recall_score(y_test, predictions, average="macro", zero_division=0))

    evaluation_metrics.log_metric("test_accuracy", accuracy)
    evaluation_metrics.log_metric("test_f1_macro", f1_macro)
    evaluation_metrics.log_metric("test_precision_macro", precision_macro)
    evaluation_metrics.log_metric("test_recall_macro", recall_macro)

    matrix = confusion_matrix(y_test, predictions, labels=[0, 1, 2]).tolist()
    classification_metrics.log_confusion_matrix(
        categories=["setosa", "versicolor", "virginica"],
        matrix=matrix
    )

    print("테스트 샘플 수:", len(y_test))
    print("Accuracy:", accuracy)
    print("Macro F1:", f1_macro)
    print("Confusion Matrix:", matrix)
    return accuracy

### 3-2. 품질 검사 Component
기준 미달이면 이 작업을 실패시켜 파이프라인 상태로 구별합니다. 이는 실습용 검증 기준이며 운영 배포 승인을 대신하지 않습니다.

In [14]:
@dsl.component(base_image="python:3.11-slim")
def quality_gate(
    test_accuracy: float,
    cv_accuracy_mean: float,
    min_test_accuracy: float = 0.85,
    min_cv_accuracy: float = 0.85
) -> float:
    print("테스트 정확도:", test_accuracy)
    print("CV 평균 정확도:", cv_accuracy_mean)
    print("테스트 기준:", min_test_accuracy)
    print("CV 기준:", min_cv_accuracy)

    if test_accuracy < min_test_accuracy:
        raise ValueError("테스트 정확도가 기준에 미달합니다.")
    if cv_accuracy_mean < min_cv_accuracy:
        raise ValueError("교차검증 평균이 기준에 미달합니다.")

    print("모델 품질 검사 통과")
    return test_accuracy

### 3-3. v3 Pipeline 정의
KFP가 출력 아티팩트를 입력에 연결해 작업 의존성을 자동으로 구성합니다.

In [15]:
@dsl.pipeline(name="iris-validated-pipeline")
def iris_validated_pipeline(
    n_estimators: int = 100,
    min_test_accuracy: float = 0.85,
    min_cv_accuracy: float = 0.85
) -> float:
    train_task = train_model_v3(n_estimators=n_estimators)

    evaluate_task = evaluate_model_v3(
        model_artifact=train_task.outputs["model_artifact"],
        test_data=train_task.outputs["test_data"]
    )

    gate_task = quality_gate(
        test_accuracy=evaluate_task.outputs["Output"],
        cv_accuracy_mean=train_task.outputs["Output"],
        min_test_accuracy=min_test_accuracy,
        min_cv_accuracy=min_cv_accuracy
    )

    return gate_task.output

### 3-4. YAML 자동 생성
컴파일은 파이프라인 명세를 파일로 저장할 뿐, Kubernetes에서 실행하지 않습니다.

In [16]:
compiler.Compiler().compile(
    pipeline_func=iris_validated_pipeline,
    package_path="iris-validated-pipeline.yaml"
)

from pathlib import Path
print("v3 YAML 생성:", Path("iris-validated-pipeline.yaml").exists())

v3 YAML 생성: True


### 3-5. v3 실행 요청 (선택적으로 실행)
아래 셀을 실행하면 실제로 새로운 KFP Run을 제출합니다. **이전 성공 기록을 덮어쓰지 않습니다.**

In [17]:
run_v3 = client.create_run_from_pipeline_package(
    pipeline_file="iris-validated-pipeline.yaml",
    experiment_name="iris-mlops-experiment",
    arguments={
        "n_estimators": 100,
        "min_test_accuracy": 0.85,
        "min_cv_accuracy": 0.85
    }
)
print("v3 Run ID:", run_v3.run_id)

v3 Run ID: bad50fdd-c5c2-4ba5-8cf6-8825ac264238


### 3-6. 실행 상태 확인
제출 직후에는 `PENDING` 또는 `RUNNING`일 수 있습니다. 잠시 후 다시 실행하면 됩니다.

In [21]:
run_v3_detail = client.get_run(run_id=run_v3.run_id)

print("Pipeline 상태:", run_v3_detail.state)
print("Run ID:", run_v3.run_id)

if run_v3_detail.state == "SUCCEEDED":
    print("v3 Pipeline 실행 성공!")
elif run_v3_detail.state == "FAILED":
    print("v3 Pipeline 실행 실패")
elif run_v3_detail.state == "RUNNING":
    print("실행 중입니다.")
else:
    print("대기 중이거나 아직 완료되지 않았습니다.")

Pipeline 상태: SUCCEEDED
Run ID: bad50fdd-c5c2-4ba5-8cf6-8825ac264238
v3 Pipeline 실행 성공!


## 실습 범위와 다음 단계

- **완료된 v1·v2:** 실제 KFP 실행 성공, Artifact 전달 및 테스트 Accuracy `1.0` 확인
- **추가된 v3:** 코드 작성 및 컴파일·실행용 셀 제공. **아직 실제 `SUCCEEDED`는 확인되지 않았습니다.**
- v3가 성공하면 UI에서 `train-model-v3`의 **Metrics**, `evaluate-model-v3`의 **Metrics/ClassificationMetrics** 아티팩트를 확인합니다. UI 지원 범위에 따라 혼동행렬은 아티팩트 메타데이터에서 확인할 수도 있습니다.
- v3에서 저장하는 모델은 **KFP의 Model Artifact**이며, **Model Registry 등록이나 MLflow 기록, KServe 배포는 아직 포함되지 않았습니다.**
- 컴파일된 `iris-validated-pipeline.yaml`은 Kubeflow Notebook에서 컴파일 셀을 실행한 뒤 생성됩니다. 이 문서만 다운로드했다고 YAML까지 자동으로 생성되지는 않습니다.
